# Supervised Classification 

pipeline:
1. embed all articles using sentence-transformers 
2. save embeddings for clustering
3. train classifiers on top of embeddings

In [3]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.preprocessing import LabelEncoder
import re
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## load clean data

from the data prep notebook

In [34]:
#load prepped datasets
df =pd.read_csv('Data/tagged_articles_clean.csv')
untagged= pd.read_csv('Data/untagged_articles_clean.csv')

print(f'tagged: {df.shape}')
print(f'untagged: {untagged.shape}')
print()
print(df['User_Needs'].value_counts())

tagged: (3805, 25)
untagged: (5037, 24)

User_Needs
update-me              1894
give-me-perspective     725
educate-me              584
connect-me              292
inspire-me              208
help-me                 102
Name: count, dtype: int64


In [35]:
#basic text cleaning
def clean_txt(text):
    text= str(text).lower()
    text =re.sub(r'[^a-z0-9\s]','', text)
    text= re.sub(r'\s+',' ',text).strip()
    return text

df['clean_text'] =df['text'].apply(clean_txt)
untagged['clean_text']= untagged['text'].apply(clean_txt)

avg_words =df['clean_text'].str.split().str.len().mean()
print(f'avg cleaned article: {avg_words:.0f} words')

avg cleaned article: 685 words


In [36]:
#encode labels
le= LabelEncoder()
df['label'] =le.fit_transform(df['User_Needs'])

label_map =dict(zip(le.classes_,le.transform(le.classes_)))
print(label_map)
num_classes =len(label_map)

{'connect-me': np.int64(0), 'educate-me': np.int64(1), 'give-me-perspective': np.int64(2), 'help-me': np.int64(3), 'inspire-me': np.int64(4), 'update-me': np.int64(5)}


---
### Naïve Bayes



In [21]:
#gaussian naive bayes on embeddings
from sklearn.naive_bayes import GaussianNB

nb_model =GaussianNB()
nb_model.fit(X_train,y_train)

nb_preds= nb_model.predict(X_test)

print('=== Naive Bayes on Embeddings ===')
print(f'accuracy: {accuracy_score(y_test,nb_preds):.4f}')
print()
print(classification_report(y_test,nb_preds, target_names=le.classes_))

=== Naive Bayes on Embeddings ===
accuracy: 0.4875

                     precision    recall  f1-score   support

         connect-me       0.31      0.38      0.34        58
         educate-me       0.35      0.50      0.41       117
give-me-perspective       0.47      0.50      0.49       145
            help-me       0.24      0.55      0.34        20
         inspire-me       0.31      0.69      0.42        42
          update-me       0.79      0.47      0.59       379

           accuracy                           0.49       761
          macro avg       0.41      0.52      0.43       761
       weighted avg       0.58      0.49      0.51       761



---


### Picks the Majority Class Everytime

In [4]:
#baseline: predict majority class for everything
majority_label =le.transform(['update-me'])[0]
dummy_preds= np.full(len(y_test),majority_label)
dummy_acc =accuracy_score(y_test,dummy_preds)
print(f'=== Always Predict update-me: {dummy_acc:.4f} ===')
print(classification_report(y_test,dummy_preds,target_names=le.classes_))

=== Always Predict update-me: 0.4980 ===
                     precision    recall  f1-score   support

         connect-me       0.00      0.00      0.00        58
         educate-me       0.00      0.00      0.00       117
give-me-perspective       0.00      0.00      0.00       145
            help-me       0.00      0.00      0.00        20
         inspire-me       0.00      0.00      0.00        42
          update-me       0.50      1.00      0.66       379

           accuracy                           0.50       761
          macro avg       0.08      0.17      0.11       761
       weighted avg       0.25      0.50      0.33       761



c:\Users\Pylab\desktop\ALL DESKTOP STUFF\UNSUPERVISED_ML\UML-FINAL-PROJECT\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Pylab\desktop\ALL DESKTOP STUFF\UNSUPERVISED_ML\UML-FINAL-PROJECT\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Pylab\desktop\ALL DESKTOP STUFF\UNSUPERVISED_ML\UML-FINAL-PROJECT\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted sa

## Gemma embeddings code

In [1]:
#calling embeddings
import pandas as pd
import numpy as np
import h5py
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch

print('loading h5 file and data...')

#load embeddings
with h5py.File("Data/semantic_embeddings.h5", "r") as f:
    class_embeddings = f["classification_embedding"][:]
    urls = [u.decode('utf-8') for u in f["URL"][:]]

print(f'classification embeddings shape: {class_embeddings.shape}')

#load clean tagged data (has labels)
df_tagged = pd.read_csv('Data/tagged_articles_clean.csv')
print(f'tagged data shape: {df_tagged.shape}')

#label encoding
le = LabelEncoder()
df_tagged['label'] = le.fit_transform(df_tagged['User_Needs'])
num_classes = len(le.classes_)
print('classes:', le.classes_)

#use classification embeddings for tagged articles (first 3805 rows should match)
tagged_embeddings = class_embeddings[:len(df_tagged)]

#train/test split
X_train, X_test, y_train, y_test = train_test_split(tagged_embeddings, df_tagged['label'].values, test_size=0.2, random_state=42, stratify=df_tagged['label'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
print('ready to train models!')

loading h5 file and data...
classification embeddings shape: (3805, 768)
tagged data shape: (3805, 25)
classes: ['connect-me' 'educate-me' 'give-me-perspective' 'help-me' 'inspire-me'
 'update-me']
device: cpu
ready to train models!


### FFNN

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight

#class weights
class_weights =compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights =torch.FloatTensor(class_weights).to(device)

class ImprovedFFNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1= nn.Linear(768, 512)
        self.fc2=nn.Linear(512, 256)
        self.fc3=nn.Linear(256, 128)
        self.fc4=nn.Linear(128, num_classes)
        self.dropout=nn.Dropout(0.35)
        self.relu =nn.ReLU()
    
    def forward(self,x):
        x =self.relu(self.fc1(x))
        x=self.dropout(x)
        x=self.relu(self.fc2(x))
        x= self.dropout(x)
        x= self.relu(self.fc3(x))
        x=self.fc4(x)
        return x

ffnn = ImprovedFFNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(ffnn.parameters(), lr=0.001, weight_decay=1e-5)

train_dataset= TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
test_dataset =TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))

train_loader=DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader= DataLoader(test_dataset, batch_size=128, shuffle=False)

print('training Improved FFNN...')
best_acc =0
epochs= 30

for epoch in range(epochs):
    ffnn.train()
    for bx,by in train_loader:
        bx,by=bx.to(device), by.to(device)
        optimizer.zero_grad()
        out =ffnn(bx)
        loss= criterion(out, by)
        loss.backward()
        optimizer.step()
    
    ffnn.eval()
    correct = total = 0
    with torch.no_grad():
        for bx,by in test_loader:
            bx= bx.to(device)
            out =ffnn(bx)
            preds=torch.argmax(out, dim=1)
            correct+= (preds.cpu()== by).sum().item()
            total+= by.size(0)
    acc=correct/total
    
    if acc>best_acc:
        best_acc =acc
        torch.save(ffnn.state_dict(), 'best_ffnn_gemma.pth')
    
    if epoch%5==0 or epoch == epochs-1:
        print(f'epoch {epoch} val acc: {acc:.4f}')

print(f'Best FFNN accuracy: {best_acc:.4f}')

training Improved FFNN...
epoch 0 val acc: 0.5769
epoch 5 val acc: 0.6518
epoch 10 val acc: 0.6728
epoch 15 val acc: 0.6610
epoch 20 val acc: 0.6938
epoch 25 val acc: 0.7004
epoch 29 val acc: 0.6767
Best FFNN accuracy: 0.7070


### CatBoost

In [2]:
#CatBoost - removed incompatible subsample parameter
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

print('training CatBoost...')

cat_model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.04,
    random_seed=42,
    verbose=0,
    auto_class_weights='Balanced',   # helps with imbalance
    bootstrap_type='MVS'             # changed to MVS to avoid error
)

cat_model.fit(X_train, y_train)

cat_preds = cat_model.predict(X_test)
cat_acc = accuracy_score(y_test, cat_preds)

print(f'CatBoost accuracy: {cat_acc:.4f}')

training CatBoost...
CatBoost accuracy: 0.7148


### XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

print('training tuned xgboost on gemma embeddings...')

xgb_model = xgb.XGBClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.025,
    subsample=0.85,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
print(f'xgboost accuracy: {xgb_acc:.4f}')

training tuned xgboost on gemma embeddings...
xgboost accuracy: 0.7109


### LightGBM

In [ ]:
import lightgbm as lgb

print('training lightgbm...')

lgb_model = lgb.LGBMClassifier( n_estimators=1200,max_depth=8,learning_rate=0.03,subsample=0.85,colsample_bytree=0.8,random_state=42,verbose=-1,n_jobs=-1)

lgb_model.fit(X_train, y_train)
lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test))
print(f'lightgbm accuracy: {lgb_acc:.4f}')

training lightgbm...
lightgbm accuracy: 0.7070


### RNN Ensemble

In [3]:
#rnn ensemble on gemma embeddings
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.metrics import accuracy_score,classification_report
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats

#class weights for imbalance
wts =compute_class_weight('balanced',classes=np.unique(y_train),y=y_train)
class_weights= torch.FloatTensor(wts).to(device)

#reshape 768 -> 12 steps x 64 features
seq_len=12;feat_dim=64
X_tr =torch.FloatTensor(X_train.reshape(-1,seq_len,feat_dim))
X_te= torch.FloatTensor(X_test.reshape(-1,seq_len,feat_dim))
y_tr =torch.LongTensor(y_train);y_te=torch.LongTensor(y_test)
tr_loader =DataLoader(TensorDataset(X_tr,y_tr),batch_size=64,shuffle=True)
te_loader= DataLoader(TensorDataset(X_te,y_te),batch_size=128)

#3 rnn models
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(feat_dim,128,batch_first=True,bidirectional=False,dropout=0.3,num_layers=2)
        self.fc=nn.Linear(128,num_classes);self.drop=nn.Dropout(0.3)
    def forward(self,x):
        _,(h,_)=self.lstm(x);return self.fc(self.drop(h[-1]))

class GRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru=nn.GRU(feat_dim,128,batch_first=True,bidirectional=False,dropout=0.3,num_layers=2)
        self.fc=nn.Linear(128,num_classes);self.drop=nn.Dropout(0.3)
    def forward(self,x):
        _,h=self.gru(x);return self.fc(self.drop(h[-1]))

class BiLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(feat_dim,128,batch_first=True,bidirectional=True,dropout=0.3,num_layers=2)
        self.fc=nn.Linear(256,num_classes);self.drop=nn.Dropout(0.3)
    def forward(self,x):
        _,(h,_)=self.lstm(x);return self.fc(self.drop(torch.cat((h[-2],h[-1]),dim=1)))

#train + predict helpers
def train_it(model,loader,epochs=25):
    model.to(device);opt=torch.optim.Adam(model.parameters(),lr=0.001)
    crit=nn.CrossEntropyLoss(weight=class_weights)
    for ep in range(epochs):
        model.train();tl=0
        for bx,by in loader:
            bx,by=bx.to(device),by.to(device);opt.zero_grad()
            loss=crit(model(bx),by);loss.backward();opt.step();tl+=loss.item()
        if (ep+1)%5==0:print(f'  ep {ep+1}/{epochs} loss:{tl/len(loader):.4f}')
    return model

def predict_it(model,loader):
    model.eval();p=[]
    with torch.no_grad():
        for bx,by in loader:
            p.extend(torch.argmax(model(bx.to(device)),dim=1).cpu().numpy())
    return np.array(p)

#train all 3
print('training LSTM...')
lstm_m=train_it(LSTMModel(),tr_loader)
print('\ntraining GRU...')
gru_m=train_it(GRUModel(),tr_loader)
print('\ntraining BiLSTM...')
bilstm_m=train_it(BiLSTMModel(),tr_loader)

#individual results
lp=predict_it(lstm_m,te_loader);gp=predict_it(gru_m,te_loader);bp=predict_it(bilstm_m,te_loader)
print(f'\nlstm:{accuracy_score(y_test,lp):.4f} gru:{accuracy_score(y_test,gp):.4f} bilstm:{accuracy_score(y_test,bp):.4f}')

#majority vote ensemble
stacked=np.stack([lp,gp,bp],axis=1)
ens_preds=stats.mode(stacked,axis=1)[0].flatten()
print(f'\n=== RNN Ensemble (Gemma): {accuracy_score(y_test,ens_preds):.4f} ===')
print(classification_report(y_test,ens_preds,target_names=le.classes_))

training LSTM...
  ep 5/25 loss:1.5543
  ep 10/25 loss:1.2371
  ep 15/25 loss:1.1309
  ep 20/25 loss:1.0712
  ep 25/25 loss:1.0009

training GRU...
  ep 5/25 loss:1.3984
  ep 10/25 loss:1.1381
  ep 15/25 loss:1.0287
  ep 20/25 loss:0.8874
  ep 25/25 loss:0.7967

training BiLSTM...
  ep 5/25 loss:1.3802
  ep 10/25 loss:1.1618
  ep 15/25 loss:1.0375
  ep 20/25 loss:0.9509
  ep 25/25 loss:0.8761

lstm:0.5887 gru:0.5125 bilstm:0.5466

=== RNN Ensemble (Gemma): 0.5519 ===
                     precision    recall  f1-score   support

         connect-me       0.36      0.57      0.44        58
         educate-me       0.28      0.45      0.34       117
give-me-perspective       0.68      0.53      0.60       145
            help-me       0.25      0.50      0.33        20
         inspire-me       0.42      0.74      0.53        42
          update-me       0.86      0.57      0.69       379

           accuracy                           0.55       761
          macro avg       0.47      0.